# Phase 3: Project-Level Analysis

This notebook queries the cleaned projects table from our SQLite database, performs analytical transformations (cost overrun %, timeline delays), analyzes the metrics across different project types, and visualizes the results using Matplotlib and Seaborn.

In [ ]:
import os
import sys
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Append src folder to path
sys.path.append(os.path.abspath('../src'))

import db_connection

## 1. Load Data from SQLite

In [ ]:
conn = db_connection.get_connection()
df_projects = pd.read_sql_query("SELECT * FROM projects", conn)
conn.close()

# Convert dates to datetime objects for calculations
df_projects["start_date"] = pd.to_datetime(df_projects["start_date"])
df_projects["planned_end_date"] = pd.to_datetime(df_projects["planned_end_date"])
df_projects["actual_end_date"] = pd.to_datetime(df_projects["actual_end_date"])

df_projects.head()

## 2. Compute Cost and Timeline Metrics

- **Cost Overrun %** = `(actual_cost - budgeted_cost) / budgeted_cost`
- **Timeline Delay (days)** = `actual_end_date - planned_end_date` (for completed projects). For ongoing/delayed projects, we can compute delay relative to today or relative to planned_end_date if it has already passed.

In [ ]:
# Cost Overrun %
df_projects["cost_variance"] = df_projects["actual_cost"] - df_projects["budgeted_cost"]
df_projects["cost_overrun_pct"] = (df_projects["cost_variance"] / df_projects["budgeted_cost"]) * 100

# Delay in Days
# For completed projects, compare actual_end_date to planned_end_date
# For delayed/ongoing projects where planned_end_date has passed, we compare to a reference date (e.g. 2026-07-18 as per metadata)
ref_date = pd.to_datetime("2026-07-18")

def calculate_delay(row):
    if pd.notna(row["actual_end_date"]):
        return (row["actual_end_date"] - row["planned_end_date"]).days
    elif row["status"] == "Delayed" or (row["status"] == "Ongoing" and ref_date > row["planned_end_date"]):
        return (ref_date - row["planned_end_date"]).days
    else:
        return 0

df_projects["delay_days"] = df_projects.apply(calculate_delay, axis=1)
df_projects.head()

## 3. Project Type Breakdown Analysis

In [ ]:
breakdown = df_projects.groupby("project_type").agg(
    total_projects=("project_id", "count"),
    avg_budget_cost=("budgeted_cost", "mean"),
    avg_actual_cost=("actual_cost", "mean"),
    avg_overrun_pct=("cost_overrun_pct", "mean"),
    avg_delay_days=("delay_days", "mean"),
    delayed_count=("status", lambda x: (x == "Delayed").sum())
).reset_index()

breakdown

## 4. Visualizations

In [ ]:
sns.set_theme(style="whitegrid")
os.makedirs("../reports", exist_ok=True)

# Plot 1: Cost Overrun Distribution by Project Type
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_projects, x="project_type", y="cost_overrun_pct", palette="Set2")
plt.title("Cost Overrun % Distribution by Project Type", fontsize=14, fontweight='bold')
plt.xlabel("Project Type", fontsize=12)
plt.ylabel("Cost Overrun %", fontsize=12)
plt.axhline(0, color='red', linestyle='--', linewidth=1.5)
plt.tight_layout()
plt.savefig("../reports/cost_overrun_distribution.png", dpi=300)
plt.show()

# Plot 2: Average Delay Days by Project Type
plt.figure(figsize=(10, 6))
sns.barplot(data=df_projects, x="project_type", y="delay_days", palette="viridis", errorbar=None)
plt.title("Average Project Delay (Days) by Project Type", fontsize=14, fontweight='bold')
plt.xlabel("Project Type", fontsize=12)
plt.ylabel("Average Delay in Days", fontsize=12)
plt.tight_layout()
plt.savefig("../reports/average_delay_days.png", dpi=300)
plt.show()

# Plot 3: Budgeted vs Actual Cost (Scatter Plot)
plt.figure(figsize=(10, 6))
max_val = max(df_projects["budgeted_cost"].max(), df_projects["actual_cost"].max())
sns.scatterplot(data=df_projects, x="budgeted_cost", y="actual_cost", hue="project_type", style="status", s=100, palette="deep")
plt.plot([0, max_val], [0, max_val], color="red", linestyle="--", label="On Budget Line")
plt.title("Budgeted vs. Actual Cost", fontsize=14, fontweight='bold')
plt.xlabel("Budgeted Cost (INR)", fontsize=12)
plt.ylabel("Actual Cost (INR)", fontsize=12)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig("../reports/budget_vs_actual_scatter.png", dpi=300)
plt.show()

# Plot 4: Cost Overrun % vs Timeline Delay (Days)
plt.figure(figsize=(10, 6))
sns.regplot(data=df_projects, x="delay_days", y="cost_overrun_pct", scatter_kws={'s':80}, line_kws={'color':'red'})
plt.title("Correlation: Cost Overrun % vs. Timeline Delay (Days)", fontsize=14, fontweight='bold')
plt.xlabel("Timeline Delay (Days)", fontsize=12)
plt.ylabel("Cost Overrun %", fontsize=12)
plt.tight_layout()
plt.savefig("../reports/cost_vs_timeline_correlation.png", dpi=300)
plt.show()